# 05 Fine-tune ModernBERT (rung C)

Run on **Google Colab with a T4 GPU** (Runtime -> Change runtime type -> T4 GPU).

- Same frozen test set as notebooks 02 and 03. Test is touched once, at the end.
- Model selection uses a 10% validation split carved out of **train**.
- Output: `modernbert_test_preds.csv` with `true, pred, confidence, latency_ms`, in test order.
  Download it into `data/` and score it with
  `python -m scripts.evaluate data/modernbert_test_preds.csv --name modernbert`.

In [ ]:
!pip -q install -U "transformers>=4.48" datasets accelerate

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

SEED = 42
MODEL = "answerdotai/ModernBERT-base"
MAX_LEN = 512   # same cut-off as bge-small, so the comparison is fair
print(torch.cuda.get_device_name(0))

In [ ]:
ds = load_dataset("liri-uzh/cfpb-complaints-mini")
train = ds["train"].to_pandas()
test = ds["test"].to_pandas()

labels = sorted(train["label"].unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

# Validation comes from train only; test stays frozen.
tr, val = train_test_split(train, test_size=0.1, stratify=train["label"], random_state=SEED)
len(tr), len(val), len(test)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def to_dataset(df):
    d = Dataset.from_pandas(df[["text", "label"]].reset_index(drop=True))
    d = d.map(lambda b: {"labels": [label2id[l] for l in b["label"]]}, batched=True)
    d = d.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN), batched=True)
    return d.remove_columns(["text", "label"])

tr_ds, val_ds, test_ds = to_dataset(tr), to_dataset(val), to_dataset(test)

# Share of complaints longer than MAX_LEN tokens (the truncation question from notebook 02)
lengths = [len(x) for x in tokenizer(test["text"].tolist())["input_ids"]]
print("test complaints over", MAX_LEN, "tokens:", round(np.mean(np.array(lengths) > MAX_LEN), 3))

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=len(labels), id2label=id2label, label2id=label2id)

def compute_metrics(eval_pred):
    logits, y = eval_pred
    return {"macro_f1": f1_score(y, logits.argmax(-1), average="macro")}

args = TrainingArguments(
    output_dir="out",
    learning_rate=5e-5,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,     # effective batch 16, fits a T4 at 512 tokens
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    save_total_limit=1,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(model=model, args=args, train_dataset=tr_ds, eval_dataset=val_ds,
                  data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)
trainer.train()

## Final evaluation on the frozen test set (run once)

In [ ]:
out = trainer.predict(test_ds)
proba = torch.softmax(torch.tensor(out.predictions), dim=-1).numpy()
pred = [id2label[i] for i in proba.argmax(-1)]

print("macro F1:", round(f1_score(test["label"], pred, average="macro"), 4))
print(classification_report(test["label"], pred))

In [ ]:
# Latency: one complaint at a time, as in production. Measured on the T4 GPU.
model.eval()
times = []
for text in test["text"].head(200):
    start = time.perf_counter()
    with torch.no_grad():
        enc = tokenizer(text, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(model.device)
        model(**enc)
    torch.cuda.synchronize()
    times.append((time.perf_counter() - start) * 1000)
print("median ms:", np.median(times), "p95 ms:", np.percentile(times, 95))

latency = np.full(len(test), np.nan)
latency[:200] = times
preds = pd.DataFrame({"true": test["label"], "pred": pred,
                      "confidence": proba.max(-1), "latency_ms": latency})
preds.to_csv("modernbert_test_preds.csv", index=False)

In [ ]:
from google.colab import files
files.download("modernbert_test_preds.csv")

## Notes

- Latency here is GPU. Serving on CPU would be much slower; measure that before choosing it.
- Cost: a T4 on Colab free tier is $0, but a production GPU is not. Count it in the comparison.